In [1]:
print(";dfnkskgd")

;dfnkskgd


In [2]:
%pwd
import os
os.chdir("../")
#Just went one step bcakword with this directory....

%pwd

'd:\\mlops\\Crypto_Guardian'

In [32]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainingConfig:
  root_dir: Path
  train_data_path: Path
  test_data_path: Path
  model_path: Path
  model_name: str
  train_embedding: Path
  test_embedding: Path
  p_c: int
  p_gamma: str
  p_kernel: str

In [33]:
from src.Crypto.constants import *
from src.Crypto.utils.helper import read_yaml,create_directories
from src.Crypto import logger

class ConfigurationManager:
    def __init__(self,config_file_path=CONFIG_FILE_PATH,
                      params_file_path = PARAMS_FILE_PATH,
                      schema_file_path = SCHEMA_FILE_PATH):
        self.config =read_yaml( config_file_path)
        self.parmas = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)
        
        # create_directories([self.config.artifacts_root])
        logger.debug(f"Till Now all the Yaml Files are Read Sucessfully...✅")
        
    def get_model_training(self)->ModelTrainingConfig:
        config = self.config.model_training
        parmas = self.parmas
        create_directories([config.root_dir])
        
        model_trainer = ModelTrainingConfig(
                    root_dir = config.root_dir,
                    train_data_path = config.train_data_path,
                    test_data_path= config.test_data_path,
                    model_path= config.model_path,
                    model_name= config.model_name,
                    p_c= parmas.SVC.C,
                    p_gamma= parmas.SVC.gamma,
                    p_kernel= parmas.SVC.kernel,
                    train_embedding= config.train_embedding,
                    test_embedding= config.test_embedding
                    
        )
        logger.debug("get_data_ngestion is working compeletely fine...✅")
        return model_trainer


In [34]:
from src.Crypto import logger
import pandas as pd
from transformers import pipeline
import numpy as np
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,f1_score, classification_report
import joblib

class ModelTrainerComponent:
    def __init__(self, config: ModelTrainingConfig):
        self.config = config
        
        # 1. Data Loading
        self.df = pd.read_csv(self.config.train_data_path)
        self.test_df = pd.read_csv(self.config.test_data_path)
        
        # 2. Embeddings load karna (Train aur Test dono ki zaroorat padegi)
        self.x_train_emb = np.load(self.config.train_embedding) 
        self.x_test_emb = np.load(self.config.test_embedding) # <--- Test embeddings bhi chahiye!
        
        self.y_train = self.df['Label']
        self.y_test = self.test_df['Label']
        
        self.smote = SMOTE(random_state=42)

    def prepare_data(self):
        # self.smote use hoga aur result ko self mein save karein taaki train function use kar sake
        self.x_resampled, self.y_resampled = self.smote.fit_resample(self.x_train_emb, self.y_train)
        logger.info(f"Oversampling complete: {Counter(self.y_resampled)}")
        
    def train_model(self):
        logger.info("Training SVM Model...")
        
        # Parameters ke naam sahi karein (C capital hota hai)
        clf = SVC(
            C=self.config.p_c,
            gamma=self.config.p_gamma, # Gamma ki spelling theek ki
            kernel=self.config.p_kernel,
            probability=True, # Recommendation: Prediction scores ke liye zaroori hai
            class_weight='balanced'
        )
        
        # Train on resampled embeddings
        clf.fit(self.x_resampled, self.y_resampled)
        
        # 3. CRITICAL FIX: Predict hamesha test EMBEDDINGS par hoga, text par nahi
        y_pred = clf.predict(self.x_test_emb)
        
        # Metrics calculation
        acc = accuracy_score(self.y_test, y_pred)
        f1 = f1_score(self.y_test, y_pred, average='weighted')
        
        logger.info(f"Test Accuracy: {acc}")
        logger.info(f"Test F1 Score: {f1}")
        
        # Model saving
        os.makedirs(os.path.dirname(self.config.model_path), exist_ok=True)
        joblib.dump(clf, self.config.model_path)
        
        logger.info(f"Model saved at: {self.config.model_path}")

In [35]:
try:
    cfm = ConfigurationManager()
    data_ingestion = cfm.get_model_training()
    data_ingestion_component = ModelTrainerComponent(data_ingestion)
    data_ingestion_component.prepare_data()
    data_ingestion_component.train_model()
    logger.info("Pipeline Ran Sucessfully...✅")
except Exception as e:
    logger.error("Pipeline Error...❌")
    raise e

[06-05-2026 01:58:47 PM - helper - INFO - Yaml File :config\config.yaml Read Sucessfully✅]
[06-05-2026 01:58:47 PM - helper - INFO - Yaml File :params.yaml Read Sucessfully✅]
[06-05-2026 01:58:47 PM - helper - INFO - Yaml File :schema.yaml Read Sucessfully✅]
[06-05-2026 01:58:47 PM - helper - INFO - Folder Created Sucessfully: artifacts/model_training]
[06-05-2026 01:58:47 PM - 1320316389 - INFO - Oversampling complete: Counter({1: 909, 0: 909})]
[06-05-2026 01:58:47 PM - 1320316389 - INFO - Training SVM Model...]
[06-05-2026 01:58:50 PM - 1320316389 - INFO - Test Accuracy: 0.7673469387755102]
[06-05-2026 01:58:50 PM - 1320316389 - INFO - Test F1 Score: 0.7640971817298348]
[06-05-2026 01:58:50 PM - 1320316389 - INFO - Model saved at: artifacts/model_training/model.joblib]
[06-05-2026 01:58:50 PM - 1078636700 - INFO - Pipeline Ran Sucessfully...✅]
